In [2]:
# %%
# =============================================================================
# 03_llm_inference.ipynb
# Financial AI Governance — LLM Inference (Baseline vs RAG)
# Kernel : Python (llm_env)
# Input  : data/processed/dataset_final.json
#          vectordb/ (Chroma persistent stores from Notebook 02)
# Output : results/responses/responses_baseline.json
#          results/responses/responses_rag.json
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# Directory paths
DATA_DIR     = '../data/processed'
VDB_DIR      = '../vectordb'
RESPONSE_DIR = '../results/responses'

os.makedirs(RESPONSE_DIR, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client     = OpenAI(api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)

print(f"[INFO] LLM model      : {LLM_MODEL}")
print(f"[INFO] Embedding model: {EMBED_MODEL}")
print(f"[INFO] Response dir   : {os.path.abspath(RESPONSE_DIR)}")


# %%
# =============================================================================
# Cell 2. Load Dataset and Vector Stores
# =============================================================================
# Load dataset
with open(os.path.join(DATA_DIR, 'dataset_final.json'), 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
print(f"[INFO] Dataset loaded: {len(df)} records")

# Load Chroma vector stores (persistent — no re-embedding needed)
VDB_CONFIG = {
    'NIST_AI_RMF'    : {'persist_dir': os.path.join(VDB_DIR, 'nist'),         'collection': 'nist_ai_rmf'},
    'KR_AI_BASIC_ACT': {'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'),'collection': 'kr_aibasicact'},
    'EU_AI_ACT'      : {'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),     'collection': 'eu_aiact'},
}

vector_stores = {}
for reg_key, cfg in VDB_CONFIG.items():
    vector_stores[reg_key] = Chroma(
        collection_name  = cfg['collection'],
        embedding_function = embeddings,
        persist_directory= cfg['persist_dir'],
    )
    print(f"  [LOAD] {reg_key:20s} → {cfg['persist_dir']}")

print("[INFO] All vector stores loaded.")


# %%
# =============================================================================
# Cell 3. Prompt Templates
# =============================================================================
# System prompt — shared for both baseline and RAG
SYSTEM_PROMPT = """You are an expert AI governance advisor specializing in financial institution AI compliance.
Your role is to support an AI Review Committee at a financial institution by providing accurate,
regulation-grounded answers to governance questions.

When answering:
1. Cite specific regulatory provisions (article numbers, section codes) where applicable.
2. Identify the governance axis: G1 (Accuracy), G2 (Safety), G3 (Transparency), or G4 (Compliance).
3. Flag high-risk scenarios and recommend human oversight where appropriate.
4. If uncertain, state limitations clearly rather than fabricating information.
5. Keep answers concise, structured, and actionable for a compliance committee."""

# Baseline prompt — no regulatory context provided
def build_baseline_prompt(question: str) -> str:
    return f"""Answer the following AI governance question for a financial institution AI Review Committee.

Question: {question}

Provide a structured, regulation-grounded answer."""


# RAG prompt — retrieved regulatory context injected
def build_rag_prompt(question: str, context: str) -> str:
    return f"""Answer the following AI governance question using the regulatory context provided below.

--- REGULATORY CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Provide a structured answer grounded in the regulatory context above.
Cite specific article numbers or section codes from the context where applicable."""


# %%
# =============================================================================
# Cell 4. Retrieval Function
# =============================================================================
def retrieve_context(question: str, regulation: str, k: int = 3) -> str:
    """
    Retrieve top-k relevant chunks from the regulation-specific Chroma store.

    Args:
        question   : Question string
        regulation : Regulation key (e.g., 'NIST_AI_RMF')
        k          : Number of chunks to retrieve

    Returns:
        Concatenated context string
    """
    if regulation not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation: {regulation}")
    docs    = vector_stores[regulation].similarity_search(question, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    return context


# %%
# =============================================================================
# Cell 5. LLM Inference Function
# =============================================================================
def call_llm(system_prompt: str, user_prompt: str,
             model: str = LLM_MODEL,
             temperature: float = 0.0,
             max_tokens: int = 1000) -> dict:
    """
    Call OpenAI Chat Completion API.

    Returns:
        dict with 'response', 'prompt_tokens', 'completion_tokens', 'total_tokens'
    """
    try:
        res = client.chat.completions.create(
            model      = model,
            temperature= temperature,
            max_tokens = max_tokens,
            messages   = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        return {
            'response'          : res.choices[0].message.content.strip(),
            'prompt_tokens'     : res.usage.prompt_tokens,
            'completion_tokens' : res.usage.completion_tokens,
            'total_tokens'      : res.usage.total_tokens,
        }
    except Exception as e:
        return {
            'response'          : f'[ERROR] {str(e)}',
            'prompt_tokens'     : 0,
            'completion_tokens' : 0,
            'total_tokens'      : 0,
        }


# %%
# =============================================================================
# Cell 6. Run Baseline Inference (No RAG)
# =============================================================================
print("[RUN] Baseline inference (no RAG context) ...")
print(f"      Model: {LLM_MODEL} | Temperature: 0.0 | Max tokens: 1000")
print(f"      Total records: {len(df)}\n")

baseline_results = []
total_tokens_bl  = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Baseline'):
    user_prompt = build_baseline_prompt(row['question'])
    result      = call_llm(SYSTEM_PROMPT, user_prompt)

    baseline_results.append({
        # Identifiers
        'id'              : row['id'],
        'scenario_id'     : row['scenario_id'],
        'regulation'      : row['regulation'],
        'function'        : row['function'],
        'difficulty'      : row['difficulty'],
        'financial_domain': row['financial_domain'],
        'risk_level'      : row['risk_level'],
        'governance_axis' : row['governance_axis'],
        # Content
        'question'        : row['question'],
        'ground_truth'    : row['ground_truth'],
        'legal_basis'     : row['legal_basis'],
        # Inference
        'condition'       : 'baseline',
        'context_used'    : '',
        'response'        : result['response'],
        # Token usage
        'prompt_tokens'   : result['prompt_tokens'],
        'completion_tokens': result['completion_tokens'],
        'total_tokens'    : result['total_tokens'],
        # Model info
        'model'           : LLM_MODEL,
        'temperature'     : 0.0,
    })

    total_tokens_bl += result['total_tokens']
    time.sleep(0.3)   # rate limit buffer

    # Progress checkpoint every 50 records
    idx = len(baseline_results)
    if idx % 50 == 0:
        errors = sum(1 for r in baseline_results if r['response'].startswith('[ERROR]'))
        avg_len = sum(len(r['response']) for r in baseline_results) / idx
        print(f"  [Checkpoint {idx:3d}/300] errors: {errors} | avg response: {avg_len:.0f} chars | tokens so far: {total_tokens_bl:,}")

# Save
out_bl = os.path.join(RESPONSE_DIR, 'responses_baseline.json')
with open(out_bl, 'w', encoding='utf-8') as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print(f"\n[SAVE] {out_bl}")
print(f"[INFO] Baseline total tokens used: {total_tokens_bl:,}")
print(f"[INFO] Estimated cost (gpt-4o-mini): ${total_tokens_bl * 0.00000015:.4f}")


# %%
# =============================================================================
# Cell 7. Run RAG Inference (With Regulatory Context)
# =============================================================================
print("[RUN] RAG inference (with regulatory context) ...")
print(f"      Model: {LLM_MODEL} | Temperature: 0.0 | Max tokens: 1000 | Top-k: 3\n")

rag_results     = []
total_tokens_rag = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='RAG'):
    # Retrieve regulation-specific context
    context     = retrieve_context(row['question'], row['regulation'], k=3)
    user_prompt = build_rag_prompt(row['question'], context)
    result      = call_llm(SYSTEM_PROMPT, user_prompt)

    rag_results.append({
        # Identifiers
        'id'              : row['id'],
        'scenario_id'     : row['scenario_id'],
        'regulation'      : row['regulation'],
        'function'        : row['function'],
        'difficulty'      : row['difficulty'],
        'financial_domain': row['financial_domain'],
        'risk_level'      : row['risk_level'],
        'governance_axis' : row['governance_axis'],
        # Content
        'question'        : row['question'],
        'ground_truth'    : row['ground_truth'],
        'legal_basis'     : row['legal_basis'],
        # Inference
        'condition'       : 'rag',
        'context_used'    : context,
        'response'        : result['response'],
        # Token usage
        'prompt_tokens'   : result['prompt_tokens'],
        'completion_tokens': result['completion_tokens'],
        'total_tokens'    : result['total_tokens'],
        # Model info
        'model'           : LLM_MODEL,
        'temperature'     : 0.0,
    })

    total_tokens_rag += result['total_tokens']
    time.sleep(0.3)   # rate limit buffer

    # Progress checkpoint every 50 records
    idx = len(rag_results)
    if idx % 50 == 0:
        errors = sum(1 for r in rag_results if r['response'].startswith('[ERROR]'))
        avg_len = sum(len(r['response']) for r in rag_results) / idx
        print(f"  [Checkpoint {idx:3d}/300] errors: {errors} | avg response: {avg_len:.0f} chars | tokens so far: {total_tokens_rag:,}")

# Save
out_rag = os.path.join(RESPONSE_DIR, 'responses_rag.json')
with open(out_rag, 'w', encoding='utf-8') as f:
    json.dump(rag_results, f, ensure_ascii=False, indent=2)

print(f"\n[SAVE] {out_rag}")
print(f"[INFO] RAG total tokens used : {total_tokens_rag:,}")
print(f"[INFO] Estimated cost (gpt-4o-mini): ${total_tokens_rag * 0.00000015:.4f}")


# %%
# =============================================================================
# Cell 8. Inference Summary
# =============================================================================
df_bl  = pd.DataFrame(baseline_results)
df_rag = pd.DataFrame(rag_results)

print("=" * 55)
print("  INFERENCE SUMMARY")
print("=" * 55)

for label, d in [('Baseline', df_bl), ('RAG', df_rag)]:
    print(f"\n[{label}]")
    print(f"  Records          : {len(d)}")
    print(f"  Avg response len : {d['response'].str.len().mean():.0f} chars")
    print(f"  Avg total tokens : {d['total_tokens'].mean():.0f}")
    print(f"  Total tokens     : {d['total_tokens'].sum():,}")
    errors = d['response'].str.startswith('[ERROR]').sum()
    print(f"  Errors           : {errors}")

print(f"\n  Grand total tokens : {total_tokens_bl + total_tokens_rag:,}")
print(f"  Total estimated cost: ${(total_tokens_bl + total_tokens_rag) * 0.00000015:.4f}")

# Response length comparison by regulation
print("\n[Response Length by Regulation (chars)]")
comp = pd.DataFrame({
    'Regulation': df_bl.groupby('regulation')['response'].apply(lambda x: x.str.len().mean()).index,
    'Baseline'  : df_bl.groupby('regulation')['response'].apply(lambda x: x.str.len().mean()).values.round(0),
    'RAG'       : df_rag.groupby('regulation')['response'].apply(lambda x: x.str.len().mean()).values.round(0),
})
print(comp.to_string(index=False))

print(f"\n✅ Notebook 03 complete — Next: 04_evaluation_g1_g4.ipynb")

[INFO] LLM model      : gpt-4o-mini
[INFO] Embedding model: text-embedding-3-small
[INFO] Response dir   : C:\Users\User\Downloads\학술\9_ai거버넌스_챗봇_ssci\financial_ai_governance\results\responses
[INFO] Dataset loaded: 300 records
  [LOAD] NIST_AI_RMF          → ../vectordb\nist
  [LOAD] KR_AI_BASIC_ACT      → ../vectordb\kr_aibasicact
  [LOAD] EU_AI_ACT            → ../vectordb\eu_aiact
[INFO] All vector stores loaded.
[RUN] Baseline inference (no RAG context) ...
      Model: gpt-4o-mini | Temperature: 0.0 | Max tokens: 1000
      Total records: 300



Baseline:  17%|███████████▊                                                           | 50/300 [09:08<41:01,  9.85s/it]

  [Checkpoint  50/300] errors: 0 | avg response: 2386 chars | tokens so far: 32,651


Baseline:  33%|███████████████████████▎                                              | 100/300 [16:44<30:37,  9.19s/it]

  [Checkpoint 100/300] errors: 0 | avg response: 2440 chars | tokens so far: 66,092


...

RAG:  83%|██████████████████████████████████████████████████████████████▌            | 250/300 [36:21<06:21,  7.62s/it]

  [Checkpoint 250/300] errors: 0 | avg response: 2690 chars | tokens so far: 502,873


RAG: 100%|███████████████████████████████████████████████████████████████████████████| 300/300 [45:26<00:00,  9.09s/it]

  [Checkpoint 300/300] errors: 0 | avg response: 2692 chars | tokens so far: 613,318



[SAVE] ../results/responses\responses_rag.json
[INFO] RAG total tokens used : 613,318
[INFO] Estimated cost (gpt-4o-mini): $0.0920
  INFERENCE SUMMARY

[Baseline]
  Records          : 300
  Avg response len : 2264 chars
  Avg total tokens : 629
  Total tokens     : 188,812
  Errors           : 0

[RAG]
  Records          : 300
  Avg response len : 2692 chars
  Avg total tokens : 2044
  Total tokens     : 613,318
  Errors           : 0

  Grand total tokens : 802,130
  Total estimated cost: $0.1203

[Response Length by Regulation (chars)]
     Regulation  Baseline    RAG
      EU_AI_ACT    2192.0 2575.0
KR_AI_BASIC_ACT    2161.0 2649.0
    NIST_AI_RMF    2440.0 2853.0

✅ Notebook 03 complete — Next: 04_evaluation_g1_g4.ipynb
